In [ ]:
import os
import time
from pathlib import Path

import whisper
import torch

from vosk import Model, KaldiRecognizer

import soundfile as sf
import json

In [ ]:
print("="*50)
print("Información del entorno")
print("="*50)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Procesamiento en CPU")

In [ ]:
#Rutas
ROOT = Path("../")

AUDIO_DIR = ROOT / "data" / "processed"

MODELS_DIR = ROOT / "models"

REFERENCE_DIR = ROOT / "data" / "references"

In [ ]:
#Cargar modelo Whisper
print("Cargando Whisper Base...")

device = "cuda" if torch.cuda.is_available() else "cpu"

whisper_model = whisper.load_model(
    "base",
    device=device
)

print("Modelo Whisper cargado correctamente.")

In [ ]:
#Importar Vosk
from vosk import Model

vosk_path = MODELS_DIR / "vosk-model-es-0.42"

vosk_model = Model(str(vosk_path))

print("Modelo Vosk cargado correctamente.")

In [ ]:
#Seleccionar un archivo de audio para prueba
audio_prueba = sorted(AUDIO_DIR.glob("*.wav"))[0]

print(audio_prueba.name)

In [ ]:
#Validar Whisper
import librosa

inicio = time.perf_counter()

audio, sr = librosa.load(
    audio_prueba,
    sr=16000,
    mono=True
)

resultado = whisper_model.transcribe(
    audio,
    language="es",
    fp16=False
)

fin = time.perf_counter()

print("="*50)
print("Whisper")
print("="*50)

print(resultado["text"])

print()

print(f"Tiempo: {fin-inicio:.3f} segundos")

In [ ]:
#Validar Vosk
data, samplerate = sf.read(audio_prueba)

inicio = time.perf_counter()

if len(data.shape) > 1:
    data = data[:,0]

rec = KaldiRecognizer(vosk_model, samplerate)

rec.AcceptWaveform(
    (data * 32767).astype("int16").tobytes()
)

resultado = json.loads(rec.FinalResult())

fin = time.perf_counter()

print("="*50)
print("Vosk")
print("="*50)

print(resultado["text"])
print(f"Tiempo de procesamiento: {fin - inicio:.2f} segundos")

In [ ]:
#Resumen
print("="*40)
print("VALIDACIÓN COMPLETADA")
print("="*40)

print("Whisper Base      OK")

print("Vosk              OK")

print("Entorno listo para procesamiento")

Actividad 2 Procesamiento de Audios

In [ ]:
#Procesamiento y transcripción de audios
import os
import json
import time
import numpy as np
import pandas as pd
import soundfile as sf

from pathlib import Path
from tqdm import tqdm

In [ ]:
#Rutas
ROOT = Path("../")

AUDIO_DIR = ROOT / "data" / "processed"

REFERENCE_FILE = ROOT / "data" / "references" / "transcripciones_referencia.csv"

RESULTS_DIR = ROOT / "results"

RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
#Leer las trasncripciones de referencia
df_ref = pd.read_csv(REFERENCE_FILE)

df_ref.head()

In [ ]:
#Procesar Whisper Base
import librosa

whisper_results = []

for _, row in tqdm(df_ref.iterrows(), total=len(df_ref)):

    audio_path = AUDIO_DIR / row["ID_Audio"]

    # Cargar audio
    audio, sr = librosa.load(
        audio_path,
        sr=16000,
        mono=True
    )

    # Medir tiempo
    inicio = time.perf_counter()

    resultado = whisper_model.transcribe(
        audio,
        language="es",
        fp16=False
    )

    fin = time.perf_counter()

    whisper_results.append({
        "ID_Audio": row["ID_Audio"],
        "Referencia": row["Texto_Corregido_Manual"],
        "Whisper": resultado["text"].strip(),
        "Latencia_Whisper": fin - inicio,
        "Duracion_s": librosa.get_duration(y=audio, sr=sr)
    })

df_whisper = pd.DataFrame(whisper_results)

df_whisper.head()

In [ ]:
#Procesar Vosk
vosk_results = []

for _, row in tqdm(df_ref.iterrows(), total=len(df_ref)):

    audio_path = AUDIO_DIR / row["ID_Audio"]

    data, samplerate = sf.read(audio_path)

    if len(data.shape) > 1:
        data = data[:,0]

    rec = KaldiRecognizer(vosk_model, samplerate)

    inicio = time.perf_counter()

    rec.AcceptWaveform(
        (data*32767).astype(np.int16).tobytes()
    )

    resultado = json.loads(rec.FinalResult())

    fin = time.perf_counter()

    vosk_results.append({

        "ID_Audio": row["ID_Audio"],

        "Vosk": resultado["text"],

        "Latencia_Vosk": fin-inicio

    })

df_vosk = pd.DataFrame(vosk_results)

df_vosk.head()

In [ ]:
#Unión de resultados
df = (

    df_whisper

    .merge(

        df_vosk,

        on="ID_Audio"

    )

)

df.head()

In [ ]:
#Guardar resultados
output = RESULTS_DIR / "transcripciones_modelos.csv"

df.to_csv(

    output,

    index=False,

    encoding="utf-8-sig"

)

print(output)

Actividad 3 Evaluación comparativa de los modelos

In [ ]:
#Calcular WER
from jiwer import wer

In [ ]:
#WER
wer_whisper = []

wer_vosk = []

for _, row in df.iterrows():

    wer_whisper.append(

        wer(

            row["Referencia"],

            row["Whisper"]

        )

    )

    wer_vosk.append(

        wer(

            row["Referencia"],

            row["Vosk"]

        )

    )

df["WER_Whisper"] = wer_whisper

df["WER_Vosk"] = wer_vosk

In [ ]:
from jiwer import process_words

In [ ]:
from jiwer import process_words

def calcular_wcpm(referencia, hipotesis, duracion_segundos):
    """
    Calcula las Palabras Correctas por Minuto (WCPM)
    utilizando la alineación de JiWER.

    Parámetros
    ----------
    referencia : str
        Transcripción de referencia.
    hipotesis : str
        Transcripción generada por el modelo.
    duracion_segundos : float
        Duración del audio en segundos.

    Retorna
    -------
    float
        WCPM
    """

    if duracion_segundos <= 0:
        return 0

    resultado = process_words(referencia, hipotesis)

    palabras_referencia = len(referencia.split())

    palabras_correctas = (
        palabras_referencia
        - resultado.substitutions
        - resultado.deletions
    )

    palabras_correctas = max(0, palabras_correctas)

    minutos = duracion_segundos / 60

    return palabras_correctas / minutos

In [ ]:
df["WCPM_Whisper"] = df.apply(
    lambda x: calcular_wcpm(
        x["Referencia"],
        x["Whisper"],
        x["Duracion_s"]
    ),
    axis=1
)

In [ ]:
df["WCPM_Vosk"] = df.apply(
    lambda x: calcular_wcpm(
        x["Referencia"],
        x["Vosk"],
        x["Duracion_s"]
    ),
    axis=1
)


In [ ]:
#Estadísticas 
resumen = pd.DataFrame({

    "Modelo":[

        "Whisper Base",

        "Vosk"

    ],

    "WER promedio":[

        df["WER_Whisper"].mean(),

        df["WER_Vosk"].mean()

    ],

    "Latencia promedio (s)":[

        df["Latencia_Whisper"].mean(),

        df["Latencia_Vosk"].mean()

    ],

    "WCPM promedio":[

        df["WCPM_Whisper"].mean(),

        df["WCPM_Vosk"].mean()

    ]

})

resumen

In [ ]:
#Grafico de Wer
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))

plt.bar(

    resumen["Modelo"],

    resumen["WER promedio"]

)

plt.title("Comparación del WER")

plt.ylabel("WER")

plt.show()

In [ ]:
#Grafico de Latencia
plt.figure(figsize=(6,4))

plt.bar(

    resumen["Modelo"],

    resumen["Latencia promedio (s)"]

)

plt.title("Comparación de Latencia")

plt.ylabel("Segundos")

plt.show()

In [ ]:
#Gráfico WCPM
plt.figure(figsize=(6,4))

plt.bar(

    resumen["Modelo"],

    resumen["WCPM promedio"]

)

plt.title("Comparación de WCPM")

plt.ylabel("Palabras Correctas por Minuto")

plt.show()

In [ ]:
#Ecuación para ponderar los pesos de la métricas
PESO_WER = 0.50
PESO_LAT = 0.20
PESO_WCPM = 0.30

assert abs(PESO_WER + PESO_LAT + PESO_WCPM - 1) < 1e-6

In [ ]:
WCPM_MAX = max(
    df["WCPM_Whisper"].max(),
    df["WCPM_Vosk"].max()
)

print(WCPM_MAX)

In [ ]:
def calcular_ida(wer, latencia, wcpm, wcpm_max):

    score_wer = 1 - wer

    score_latencia = 1 / (1 + latencia)

    score_wcpm = wcpm / wcpm_max

    ida = (
        PESO_WER * score_wer
        + PESO_LAT * score_latencia
        + PESO_WCPM * score_wcpm
    )

    return ida

In [ ]:
df["IDA_Whisper"] = df.apply(
    lambda x: calcular_ida(
        x["WER_Whisper"],
        x["Latencia_Whisper"],
        x["WCPM_Whisper"],
        WCPM_MAX
    ),
    axis=1
)

In [ ]:
df["IDA_Vosk"] = df.apply(
    lambda x: calcular_ida(
        x["WER_Vosk"],
        x["Latencia_Vosk"],
        x["WCPM_Vosk"],
        WCPM_MAX
    ),
    axis=1
)

In [ ]:
resumen_ida = pd.DataFrame({

    "Modelo": [
        "Whisper Base",
        "Vosk"
    ],

    "IDA Promedio": [
        df["IDA_Whisper"].mean(),
        df["IDA_Vosk"].mean()
    ]
})

resumen_ida

In [ ]:
#Desciación Estandar
resumen_ida = pd.DataFrame({

    "Modelo": [
        "Whisper Base",
        "Vosk"
    ],

    "IDA Promedio": [
        df["IDA_Whisper"].mean(),
        df["IDA_Vosk"].mean()
    ],

    "Desv. Estándar": [
        df["IDA_Whisper"].std(),
        df["IDA_Vosk"].std()
    ]

})

resumen_ida

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))

plt.bar(
    resumen_ida["Modelo"],
    resumen_ida["IDA Promedio"]
)

plt.ylabel("IDA")

plt.title("Índice de Desempeño Global de Modelos ASR")

plt.ylim(0,1)

plt.grid(axis="y", alpha=0.3)

plt.show()

In [ ]:
import shutil
import subprocess

print("FFmpeg:", shutil.which("ffmpeg"))
print("FFprobe:", shutil.which("ffprobe"))

subprocess.run(
    ["ffmpeg", "-version"],
    check=True
)